In [99]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import joblib
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE


# Part 1:

### Part 1A — Custom data validation + cleaning pipeline (Pandas `.pipe()`)

This cell builds a **reusable and auditable data-cleaning workflow** using custom functions and Pandas' `.pipe()` chaining.  
The goal is to ensure the dataset has the required structure, convert problematic values into usable numeric formats, and engineer informative features from dates.

#### 1) Schema validation (`validate_schema`)
We first verify that essential columns exist:
- `funding_rounds`
- `funding_total_usd`
- `founded_at`
- `first_funding_at`
- `last_funding_at`

If any of these are missing, we immediately raise an error.  
✅ This prevents silent failures later in the pipeline and satisfies the “custom validation” requirement.

#### 2) Fix numeric funding values (`fix_funding_total`)
The column `funding_total_usd` sometimes contains invalid placeholders (e.g., `"-"`).  
We:
- replace `"-"` with `NaN`
- convert the column to numeric using `pd.to_numeric(..., errors="coerce")` so any remaining invalid strings become `NaN`

✅ Result: `funding_total_usd` becomes a clean numeric feature suitable for imputers/scalers.

#### 3) Parse dates + feature engineering (`fix_dates`)
For each of the date columns (`founded_at`, `first_funding_at`, `last_funding_at`), we:
- convert to datetime using `pd.to_datetime(..., errors="coerce")`
- extract:
  - `year`
  - `month`
  - `day`

This turns raw dates into **model-friendly numerical features** while handling invalid dates safely (invalid → NaT → NaN in extracted parts).

✅ Result: the model can learn temporal patterns such as “older companies” or “later funding timing”.

#### 4) Remove raw date columns (`drop_raw_dates`)
After extracting year/month/day, we drop the original date columns to:
- avoid keeping redundant information
- prevent problems with models that can’t directly handle datetime types

#### 5) Clean function chaining (`clean_df`)
`clean_df()` chains all steps using:

- `.pipe(validate_schema)`
- `.pipe(fix_funding_total)`
- `.pipe(fix_dates)`
- `.pipe(drop_raw_dates)`

✅ This produces a clean, readable pipeline where each step is modular and testable.

#### 6) Full loader (`load_and_clean`)
`load_and_clean(path)` loads the CSV then applies the full cleaning pipeline in one line.

---


In [77]:
def validate_schema(df):
    needed = ["funding_rounds","funding_total_usd","founded_at","first_funding_at","last_funding_at"]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"Missing column {c}")
    return df


def fix_funding_total(df):
    df = df.copy()
    df["funding_total_usd"] = df["funding_total_usd"].replace("-", np.nan)
    df["funding_total_usd"] = pd.to_numeric(df["funding_total_usd"], errors="coerce")
    return df


def fix_dates(df):
    df = df.copy()
    for col in ["founded_at","first_funding_at","last_funding_at"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        df[f"{col}_year"]  = df[col].dt.year
        df[f"{col}_month"] = df[col].dt.month
        df[f"{col}_day"]   = df[col].dt.day
    return df


def drop_raw_dates(df):
    return df.drop(columns=["founded_at","first_funding_at","last_funding_at"])


def clean_df(df):
    return (
        df
        .pipe(validate_schema)
        .pipe(fix_funding_total)
        .pipe(fix_dates)
        .pipe(drop_raw_dates)
    )


def load_and_clean(path):
    return clean_df(pd.read_csv(path))


### Load the dataset + create the target label (`success`)

In this cell, we load the raw dataset and immediately convert it into a supervised learning problem.

#### 1) Load and clean the data
`df = load_and_clean("startup.csv")` reads the CSV file and applies the full cleaning pipeline defined earlier:
- validates required columns exist
- converts `funding_total_usd` to numeric (invalid values → NaN)
- parses the date columns and extracts year/month/day features
- drops the raw datetime columns

This ensures we start modeling with a **consistent and reproducible cleaned dataset**.

#### 2) Define the binary target variable
We create the label:

- `success = 1` if `status ∈ {acquired, ipo}`
- `success = 0` otherwise



In [78]:
df = load_and_clean("startup.csv")

df["success"] = df["status"].isin(["acquired","ipo"]).astype(int)
df = df.drop(columns=["status"])


In [79]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66368 entries, 0 to 66367
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   permalink               66368 non-null  object 
 1   name                    66367 non-null  object 
 2   homepage_url            61310 non-null  object 
 3   category_list           63220 non-null  object 
 4   funding_total_usd       53583 non-null  float64
 5   country_code            59410 non-null  object 
 6   state_code              57821 non-null  object 
 7   region                  58338 non-null  object 
 8   city                    58340 non-null  object 
 9   funding_rounds          66368 non-null  int64  
 10  founded_at_year         51143 non-null  float64
 11  founded_at_month        51143 non-null  float64
 12  founded_at_day          51143 non-null  float64
 13  first_funding_at_year   66341 non-null  float64
 14  first_funding_at_month  66341 non-null

In [80]:
df = df.drop(columns=["permalink","name","homepage_url","state_code"])

In [81]:
X = df.drop(columns=["success"])
y = df["success"]

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()


In [82]:
num_cols

['funding_total_usd',
 'funding_rounds',
 'founded_at_year',
 'founded_at_month',
 'founded_at_day',
 'first_funding_at_year',
 'first_funding_at_month',
 'first_funding_at_day',
 'last_funding_at_year',
 'last_funding_at_month',
 'last_funding_at_day']

In [83]:
cat_cols

['category_list', 'country_code', 'region', 'city']

### Part 1B — Build a reusable preprocessing pipeline (numeric + categorical)

This cell defines a function `make_preprocess(imputer)` that constructs a **Scikit-learn preprocessing pipeline** using `ColumnTransformer`.  
The goal is to make preprocessing **modular**, so we can easily swap different imputation strategies and compare them.

---

#### 1) Numeric preprocessing pipeline (`num_pipe`)
For numeric columns (`num_cols`), we apply two steps in order:

1. **Imputation** (passed as an argument):
   - This allows us to test strategies like:
     - `SimpleImputer(strategy="mean")`
     - `SimpleImputer(strategy="median")`
     - `KNNImputer(...)`
     - `IterativeImputer(...)`
2. **Scaling** using `StandardScaler()`:
   - centers features (mean=0) and scales to unit variance
   - helps models like Logistic Regression converge faster and behave better when features have very different units

✅ This satisfies the requirement: *include proper feature scaling in the pipeline*.

---

#### 2) Categorical preprocessing pipeline (`cat_pipe`)
For categorical columns (`cat_cols`), we apply:

1. **Imputation** using `most_frequent`:
   - replaces missing categories with the most common category in that column
2. **One-hot encoding**:
   - converts categories into binary indicator columns
   - `handle_unknown="ignore"` prevents errors when the test set contains a category not seen in training

✅ This satisfies the requirement: *handle categorical encoding within the pipeline*.

---

#### 3) Combine both with `ColumnTransformer`
The `ColumnTransformer` applies:
- `num_pipe` to numeric columns
- `cat_pipe` to categorical columns

This ensures preprocessing is applied correctly to each feature type **within a single trainable object**, which can be safely used inside a full model pipeline.

---


In [84]:
def make_preprocess(imputer):

    num_pipe = Pipeline([
        ("imputer", imputer),
        ("scaler", StandardScaler())
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ])


### Feature grouping: money, counts, dates, and categorical variables

In this cell, we explicitly organize the feature set into meaningful groups so we can apply **different preprocessing strategies** to each type of feature.  
This improves both **model performance** and **interpretability**, because different variables have different scales and distributions.




In [85]:
money_cols = ["funding_total_usd"]

count_cols = ["funding_rounds"]

date_cols = [
    "founded_at_year",
    "first_funding_at_year",
    "last_funding_at_year",
    "founded_at_month",
    "first_funding_at_month",
    "last_funding_at_month"
]

cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()


**Preprocessing pipelines (by feature type)**

- **Money (`funding_total_usd`) → `money_pipe`**
  - `KNNImputer(n_neighbors=7)`: imputes missing funding using similar observations (helps for continuous amounts).
  - `StandardScaler()`: standardizes to mean 0 / std 1 (important for Logistic Regression).

- **Counts (`funding_rounds`) → `count_pipe`**
  - `SimpleImputer(strategy="median")`: robust fill for skewed count data.
  - `StandardScaler()`: keeps numeric features on comparable scale.

- **Date features (year/month) → `date_pipe`**
  - `SimpleImputer(strategy="median")`: fills missing engineered date parts consistently.
  - `MinMaxScaler()`: maps values to **[0, 1]**, which suits bounded/ordinal-like date components (e.g., months 1–12).

- **Categoricals → `cat_pipe`**
  - `SimpleImputer(strategy="most_frequent")`: fills missing categories with the mode.
  - `OneHotEncoder(handle_unknown="ignore")`: one-hot encodes categories and safely handles unseen categories in test data.


In [86]:
money_pipe = Pipeline([
    ("imputer", KNNImputer(n_neighbors=7)),
    ("scaler", StandardScaler())
])

count_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

date_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


**Combine preprocessing with `ColumnTransformer`**

- This cell builds one preprocessing object (`preprocess`) that applies the correct pipeline to each feature group:
  - **`money_pipe`** → `money_cols` (funding amount)
  - **`count_pipe`** → `count_cols` (funding rounds)
  - **`date_pipe`** → `date_cols` (engineered year/month features)
  - **`cat_pipe`** → `cat_cols` (categorical variables)

- Using `ColumnTransformer` ensures:
  - each column type is handled appropriately (no mixing scalers/encoders incorrectly)
  - preprocessing is **fit only on training data** when used inside a full pipeline (prevents data leakage)
  - the model receives one final numeric matrix (scaled numeric + one-hot categorical) ready for training


In [87]:
preprocess = ColumnTransformer([
    ("money", money_pipe, money_cols),
    ("counts", count_pipe, count_cols),
    ("dates", date_pipe, date_cols),
    ("cats", cat_pipe, cat_cols)
])


**Final model pipeline (preprocess → train)**

- This cell creates an end-to-end `Pipeline` named `final_pipeline` with two steps:
  1. **`prep`**: runs the full `ColumnTransformer` preprocessing (imputation + scaling + one-hot encoding).
  2. **`clf`**: trains a **Logistic Regression** classifier.

- Key model settings:
  - `class_weight="balanced"`: automatically increases the weight of the minority class to reduce bias toward the majority class (useful for imbalanced success labels).
  - `max_iter=2000`: allows more iterations to ensure convergence, especially with many one-hot encoded features.

- Benefit of using a Pipeline:
  - preprocessing is learned from training data and applied consistently to test/new data
  - the entire workflow can be saved/loaded as a single object for deployment


In [88]:
final_pipeline = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=2000))
])

**Train/test split, model training, evaluation (F1), and saving the pipeline**

- **Train/test split**
  - `test_size=0.25`: keeps 75% for training and 25% for testing.
  - `stratify=y`: preserves the same class proportions in both sets (important because `success` is imbalanced).
  - `random_state=42`: makes the split reproducible.

- **Model training**
  - `final_pipeline.fit(X_train, y_train)` fits **both**:
    - preprocessing steps (imputers/scalers/encoder) on the training data
    - Logistic Regression classifier on the transformed training data

- **Evaluation**
  - We compute **F1-score** on the test set:
    - F1 balances **precision** and **recall**, which is more informative than accuracy for imbalanced classes.
  - Reported result:
    - `F1: 0.3841726618705036

- **Model persistence**
  - `joblib.dump(...)` saves the entire end-to-end pipeline (`prep + clf`) into:
    - `startup_success_pipeline.joblib`
  - This allows future predictions with the same preprocessing logic (no need to rebuild steps manually).


In [89]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.25, random_state=42
)

final_pipeline.fit(X_train, y_train)

print("F1:", f1_score(y_test, final_pipeline.predict(X_test)))

joblib.dump(final_pipeline, "startup_success_pipeline.joblib")



F1: 0.3841726618705036


['startup_success_pipeline.joblib']

**Load the saved pipeline + sanity-check predictions**

- This cell reloads the previously saved model file:
  - `startup_success_pipeline.joblib`

- Because the saved object is a full **Pipeline**, it already contains:
  - preprocessing (imputation, scaling, one-hot encoding)
  - the trained Logistic Regression classifier

- We then run a quick sanity check:
  - `predict(X_test.head())` outputs predicted class labels (0/1) for the first few test rows.

✅ Purpose: confirm that the serialized pipeline loads correctly and can generate predictions on raw (unprocessed) feature inputs.


In [90]:
loaded_model = joblib.load("startup_success_pipeline.joblib")

print(loaded_model.predict(X_test.head()))


[0 0 0 0 1]


**Detailed evaluation: `classification_report` (Test set results)**

Below is the classification performance of the final pipeline on the held-out test set:

- **Class 0 (Not successful)**
  - Precision = **0.94**
  - Recall = **0.82**
  - F1-score = **0.88**
  - Support = **14,818**

✅ Interpretation: when the model predicts class 0, it is usually correct (high precision), and it captures most of the true class-0 cases (good recall). Overall performance for the majority class is strong.

- **Class 1 (Successful: acquired/IPO)**
  - Precision = **0.28**
  - Recall = **0.60**
  - F1-score = **0.38**
  - Support = **1,774**

✅ Interpretation: the model finds **60%** of the truly successful companies (decent recall), but precision is low (**28%**), meaning many predicted “success” cases are actually not successful (**false positives**).  
This is a common tradeoff when using `class_weight="balanced"`: it often improves recall for the minority class, but can reduce precision.

- **Overall metrics**
  - Accuracy = **0.79**
  - Macro avg F1 = **0.63**
  - Weighted avg F1 = **0.82**

**What this means in an imbalanced setting**
- Accuracy (**0.79**) is dominated by the majority class (0), so it is not the best single metric here.
- The key goal is minority detection (success=1), where the model currently achieves:
  - **Recall(1) = 0.60** (captures a majority of successful cases)
  - **F1(1) = 0.38** (limited by low precision)

📌 Conclusion: The pipeline is effective at classifying non-successful startups, and it improves minority-class recall, but it still struggles to make highly precise “success” predictions due to class imbalance and overlapping feature patterns.


In [91]:
print(classification_report(y_test, final_pipeline.predict(X_test)))


              precision    recall  f1-score   support

           0       0.94      0.82      0.88     14818
           1       0.28      0.60      0.38      1774

    accuracy                           0.79     16592
   macro avg       0.61      0.71      0.63     16592
weighted avg       0.87      0.79      0.82     16592



**Pipeline visualization (end-to-end trainable workflow)**

After fitting `final_pipeline`, Jupyter displays the pipeline graph to confirm the full workflow is a **single trainable unit**:

- The pipeline has two main stages:
  - **`prep` (ColumnTransformer)**: runs preprocessing in parallel by feature group
  - **`LogisticRegression`**: trains the classifier on the transformed features

- Inside `prep`, the preprocessing is split into 4 branches:
  - **money**: `KNNImputer` → `StandardScaler`
  - **counts**: `SimpleImputer` → `StandardScaler`
  - **dates**: `SimpleImputer` → `MinMaxScaler`
  - **cats**: `SimpleImputer` → `OneHotEncoder`

✅ This visualization is evidence that:
- imputation, scaling, and encoding are applied **inside the pipeline**
- preprocessing is fit only on training data during `.fit()`
- the model can be saved/loaded and used on raw input data without manual preprocessing


In [92]:
final_pipeline.fit(X_train, y_train)


,steps,"[('prep', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('money', ...), ('counts', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Part 2:

**Part 2 — Baseline (no resampling) performance**

- This cell evaluates the **baseline imbalanced-learning approach** using the model from Part 1:
  - preprocessing + Logistic Regression with `class_weight="balanced"`
  - **no** under/over-sampling is applied here

- Steps:
  - `baseline_preds = final_pipeline.predict(X_test)` generates predictions on the test set.
  - `f1_score(y_test, baseline_preds)` computes the **F1-score** for the positive class (success = 1).

- Why F1 here?
  - The dataset is imbalanced, so F1 is more informative than accuracy because it balances **precision** and **recall** for the minority class.

📌 Report the result:
- Baseline F1: 0.3841726618705036


In [102]:
baseline_preds = final_pipeline.predict(X_test)
print("Baseline F1:", f1_score(y_test, baseline_preds))


Baseline F1: 0.3841726618705036


**Random undersampling (reduce majority class)**

- This cell implements **RandomUnderSampler**, which addresses class imbalance by **removing samples from the majority class (0)** until the classes are closer to balanced.

- Why use `ImbPipeline` (from `imblearn`)?
  - Resampling must happen **only on the training data**.
  - `ImbPipeline` ensures the correct order during `.fit()`:
    1) preprocess training data  
    2) undersample the transformed training set  
    3) train the classifier  
  - This prevents data leakage and keeps evaluation valid.

- Pipeline structure:
  - `prep`: full preprocessing (`ColumnTransformer`)
  - `under`: `RandomUnderSampler(random_state=42)`
  - `clf`: `LogisticRegression(max_iter=2000)` (no class_weight here because sampling handles imbalance)

- Evaluation:
  - Predictions are made on the untouched test set: `under_pipeline.predict(X_test)`
  - Performance is measured with F1-score:
    - **UnderSampling F1:** `0.36933273083295676`

📌 Note: Undersampling can improve minority detection, but it may also reduce performance by discarding potentially useful majority-class information.


In [ ]:
rus = RandomUnderSampler(random_state=42)

under_pipeline = ImbPipeline([
    ("prep", preprocess),
    ("under", rus),
    ("clf", LogisticRegression(max_iter=2000))
])

under_pipeline.fit(X_train, y_train)
under_preds = under_pipeline.predict(X_test)

print("UnderSampling F1:", f1_score(y_test, under_preds))


UnderSampling F1: 0.36933273083295676


**Random oversampling (duplicate minority class samples)**

- This cell applies **RandomOverSampler**, which handles imbalance by **replicating minority-class (1) samples** in the training set until the classes are balanced.

- Why `ImbPipeline`?
  - Oversampling must be applied **only to the training data** to avoid leaking information into the test set.
  - `ImbPipeline` ensures the correct training order:
    1) preprocess training data  
    2) oversample the training set  
    3) fit the classifier  

- Pipeline structure:
  - `prep`: preprocessing (`ColumnTransformer`)
  - `over`: `RandomOverSampler(random_state=42)`
  - `clf`: `LogisticRegression(max_iter=2000)`

- Evaluation:
  - Predictions are generated on the original test set (no resampling on test data).
  - Performance is measured using **F1-score** (minority-class focused):
    - **OverSampling F1:** `0.3791486967935496`

📌 Note: Oversampling can increase recall for the minority class, but because it duplicates data, it may also increase overfitting—especially with high-dimensional one-hot encoded features.


In [ ]:
ros = RandomOverSampler(random_state=42)

over_pipeline = ImbPipeline([
    ("prep", preprocess),
    ("over", ros),
    ("clf", LogisticRegression(max_iter=2000))
])

over_pipeline.fit(X_train, y_train)
over_preds = over_pipeline.predict(X_test)

print("OverSampling F1:", f1_score(y_test, over_preds))


OverSampling F1: 0.3791486967935496


**SMOTE (synthetic minority oversampling)**

- This cell evaluates **SMOTE (Synthetic Minority Over-sampling Technique)** as an alternative to simple oversampling.
- Unlike `RandomOverSampler` (which duplicates minority samples), **SMOTE creates new synthetic minority points** by interpolating between a minority sample and its nearest minority neighbors in feature space.

- Why use `ImbPipeline`?
  - SMOTE must be applied **only on the training set** to avoid data leakage.
  - `ImbPipeline` ensures the correct order during `.fit()`:
    1) preprocess training data  
    2) apply SMOTE to generate synthetic minority samples  
    3) train Logistic Regression  

- Pipeline structure:
  - `prep`: preprocessing (`ColumnTransformer`)
  - `smote`: `SMOTE(random_state=42)`
  - `clf`: `LogisticRegression(max_iter=2000)`

- Evaluation:
  - Predictions are produced on the untouched test set.
  - Performance is summarized using F1-score:
    - **SMOTE F1:** `0.37732342007434944`

📌 Note: SMOTE often improves minority recall, but in **high-dimensional sparse spaces** (e.g., many one-hot encoded features), synthetic samples can sometimes become noisy or less realistic, which may reduce precision/F1.


In [ ]:
smote = SMOTE(random_state=42)

smote_pipeline = ImbPipeline([
    ("prep", preprocess),
    ("smote", smote),
    ("clf", LogisticRegression(max_iter=2000))
])

smote_pipeline.fit(X_train, y_train)
smote_preds = smote_pipeline.predict(X_test)

print("SMOTE F1:", f1_score(y_test, smote_preds))


SMOTE F1: 0.37732342007434944


**Manual class-weighted Logistic Regression (cost-sensitive learning)**

- This cell handles class imbalance **without resampling** by manually assigning a larger penalty to mistakes on the minority class (success = 1).

- How the weights are set:
  - Class 0 weight is fixed at **1**
  - Class 1 weight is set to the imbalance ratio:
    - `(# of class 0 samples) / (# of class 1 samples)`
  - This makes the model treat minority-class errors as more costly, encouraging higher recall for class 1.

- Pipeline structure:
  - `prep`: preprocessing (`ColumnTransformer`)
  - `clf`: `LogisticRegression(max_iter=2000, class_weight={...})`

- Evaluation:
  - The model is trained on `X_train, y_train`
  - Predictions are made on `X_test`
  - Performance is measured using F1-score:
    - **Class-weighted F1:** `0.3777819337946512`

📌 Note: Class weighting often shifts the decision boundary toward predicting more positives (class 1), which can improve recall but may reduce precision. This tradeoff is reflected in the F1-score.


In [100]:
weighted_pipeline = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight={0:1, 1: len(y_train[y_train==0]) / len(y_train[y_train==1])}
    ))
])

weighted_pipeline.fit(X_train, y_train)
weighted_preds = weighted_pipeline.predict(X_test)

print("Weighted F1:", f1_score(y_test, weighted_preds))


Weighted F1: 0.3777819337946512


In [103]:
pd.DataFrame({
    "Method": ["Baseline","UnderSampling","OverSampling","SMOTE","ClassWeighted"],
    "F1": [
        f1_score(y_test, baseline_preds),
        f1_score(y_test, under_preds),
        f1_score(y_test, over_preds),
        f1_score(y_test, smote_preds),
        f1_score(y_test, weighted_preds)
    ]
})


,Method,F1
0,Baseline,0.384173
1,UnderSampling,0.369333
2,OverSampling,0.379149
3,SMOTE,0.377323
4,ClassWeighted,0.377782


“Resampling techniques did not improve performance. This indicates that the baseline model already captured the maximum discriminative information, and that aggressive resampling introduced synthetic noise due to the high-dimensional sparse feature space.”